# 01 — HMM on the monkey working-memory dataset (DANDI 000620 · FEF + DMFC) — delay-aligned

The **essential** single-session HMM pipeline, mirroring the mouse [`02_hmm_triplet.ipynb`](02_hmm_triplet.ipynb):
load → tensor → **ARI state-stability** → fit → **state ↔ task-phase** → read-outs. Two frontal areas
recorded simultaneously: **FEF** and **DMFC**, fit separately. One session (**sub-Elgar**; the pickle
has 14 usable sessions and a 2nd animal is coming — pool later).

**The task has 6 phases:** fixation → stimulus → **delay** → cue → response → reward (the 7 columns
of `trial_times` are the phase onsets). Trials are aligned to **DELAY ONSET** (t = 0, the bold dashed
line): fixation+stimulus sit at negative times and the **working-memory delay period** `[0, ~+1.0] s`
is front-and-centre (shaded). Window `[-1.4, +2.2] s` @ 100 ms — delay alignment keeps ~all 414 trials.

Poisson HMM on **raw counts** (never smoothed); `bias_startprob=False` (fixation is a nominal, not
imposed, baseline). Colours: Okabe-Ito states, cividis heatmaps — colour-vision-safe.

## Part 0 — Setup

In [ ]:
import sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.monkey import load_monkey_aligned, PHASE_NAMES
from src.dynamics.hmm import (fit_poisson_hmm, stability_scan, switch_rate,
    net_flux, state_phase_alignment)
from src.viz.style import set_style, state_colors, savefig, figure_path, stroke, SEQUENTIAL, DIVERGING
set_style()

SESSION = "ses-2022-08-20"          # 43 FEF + 56 DMFC units; sub-Elgar
AREAS = ["FEF", "DMFC"]
SEED = 0
N_ITER = 50
PHASE = "monkey_hmm"

## Part 1 — Load one session, delay-aligned

`load_monkey_aligned` re-zeroes every trial at its **delay onset** and bins the spikes into raw
counts `(trials, bins, neurons)`. `events` holds all 7 phase onsets per trial, delay-relative.

In [ ]:
mk = load_monkey_aligned(SESSION)          # delay-aligned, window [-1.4,+2.2]s @ 100 ms
bin_centers = mk["bin_centers"]
tensors = mk["tensors"]
events = mk["events"]                      # (n_trials, 7) event onsets, DELAY-relative (delay=0)
n_trials = mk["n_trials"]
BIN_S = mk["window"][2]

# phase-onset times (delay-relative medians): stimulus, delay(=0), cue, response, reward
ev_med = np.nanmedian(events, axis=0)
ONSETS = {"stimulus": ev_med[1], "delay": ev_med[2], "cue": ev_med[3],
          "response": ev_med[4], "reward": ev_med[5]}

def phase_lines(ax, top=1.03, shade=True):
    if shade:
        ax.axvspan(ONSETS["delay"], ONSETS["cue"], color="0.92", zorder=0)   # the WM delay period
    for name, t in ONSETS.items():
        bold = name == "delay"
        ax.axvline(t, color="black" if bold else "0.55", ls="--", lw=2.0 if bold else 1.0, zorder=1)
        ax.text(t, top, name, rotation=90, va="bottom", ha="center", fontsize=7,
                color="black" if bold else "0.45")

print(f"kept {n_trials} trials; {len(bin_centers)} bins; window "
      f"[{bin_centers[0]:.2f}, {bin_centers[-1]:.2f}] s from DELAY onset")
print("phase onsets (delay-relative):", {k: round(float(v), 2) for k, v in ONSETS.items()})
for a in AREAS:
    print(f"  {a}: {tensors[a].shape}  (trials, bins, units)")

## Part 2 — A sanity look (delay-aligned)

Population PSTH per area with the phase lines; the bold line + grey band is the delay / WM period.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, a in zip(axes, AREAS):
    psth = tensors[a].mean(axis=(0, 2)) / BIN_S
    sem = tensors[a].mean(axis=2).std(axis=0) / np.sqrt(n_trials) / BIN_S
    ax.plot(bin_centers, psth, lw=1.8, path_effects=stroke())
    ax.fill_between(bin_centers, psth - sem, psth + sem, alpha=0.25, lw=0)
    phase_lines(ax, top=psth.max()*1.02, shade=True)
    ax.set(title=f"{a}: population PSTH (n={tensors[a].shape[2]})",
           xlabel="time from DELAY onset (s)", ylabel="rate (sp/s)")
fig.tight_layout(); savefig(fig, PHASE, f"{SESSION}_psth"); plt.show()

## Part 3 — Choose K by state stability (ARI)

Same criterion as the mouse: held-out likelihood cannot choose K for a Poisson HMM (it climbs
forever), so we measure **reproducibility** — fit two HMMs on disjoint trial thirds, decode both on a
third, score agreement (ARI). Read the best **non-trivial** (K ≥ 3) value; K = 2 is the trivial split.

In [ ]:
STAB_KS = [2, 3, 4, 5, 6]
K_PER_AREA = {}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for j, a in enumerate(AREAS):
    t0 = time.time()
    st = stability_scan(tensors[a], STAB_KS, n_repeats=3, n_iter=20, seed=SEED, n_restarts=1)
    K = st.n_states.to_numpy(); ari = st.ari_mean.to_numpy(); nt = K >= 3
    peak = int(K[nt][np.argmax(ari[nt])]); K_PER_AREA[a] = peak
    ax[j].errorbar(K, ari, yerr=st.ari_std, marker="o", ms=5, capsize=3, lw=1.8, path_effects=stroke())
    ax[j].scatter([peak], [ari[list(K).index(peak)]], s=170, facecolors="none", edgecolors="k", lw=2.2)
    ax[j].axvspan(1.6, 2.4, color="0.9", zorder=0)
    ax[j].set(xlabel="K", ylabel="Adjusted Rand Index", xticks=STAB_KS,
              title=f"{a}: state stability (best K>=3 = {peak})")
    print(f"{a}: best non-trivial K = {peak} (ARI={ari[nt].max():.3f}) [{time.time()-t0:.0f}s]")
fig.suptitle("Choose K by REPRODUCIBILITY (circled = best non-trivial; K=2 grey = trivial split)", y=1.02)
fig.tight_layout(); savefig(fig, PHASE, f"{SESSION}_stability"); plt.show()
print("K_PER_AREA:", K_PER_AREA)

## Part 4 — Fit the HMM, and look at the states across the trial

States ordered by occupancy centre-of-mass. *Left* — Viterbi state per (trial, time), trials sorted
by when they first leave their start state. *Middle* — occupancy `P(state|time)` with the phase lines
(bold = delay onset, grey = WM delay period): a persistent state held **across the delay** and a
switch at the response is the working-memory signature. *Right* — net transition flux.

In [ ]:
baseline_mask = bin_centers < ONSETS["stimulus"]      # fixation = nominal baseline (not imposed)

def first_change(row):
    d = np.flatnonzero(row != row[0]); return d[0] if len(d) else len(row)

results = {}
for a in AREAS:
    res = fit_poisson_hmm(tensors[a], K_PER_AREA[a], baseline_mask, n_iter=N_ITER, seed=SEED,
                          bias_startprob=False)
    res.bin_centers = bin_centers
    results[a] = res
    print(f"{a}: K={K_PER_AREA[a]} converged={res.model.monitor_.converged} "
          f"forward_fraction={res.forward_fraction:.3f} switch_rate={switch_rate(res.states):.2f}/trial")

def plot_hmm(a, res):
    bc, K = res.bin_centers, res.occupancy.shape[0]; cols = state_colors(K)
    order = np.argsort([first_change(r) for r in res.states])
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    im0 = ax[0].imshow(res.states[order], aspect="auto", interpolation="nearest",
                       extent=[bc[0], bc[-1], len(order), 0], cmap=SEQUENTIAL, vmin=0, vmax=K-1)
    ax[0].axvline(0, color="w", ls="--", lw=1.5)
    ax[0].set(xlabel="time from DELAY onset (s)", ylabel="trial (sorted by 1st change)",
              title=f"{a}: HMM state (K={K})")
    fig.colorbar(im0, ax=ax[0], label="state", ticks=range(K))
    for s in range(K):
        ax[1].plot(bc, res.occupancy[s], color=cols[s], lw=2.2, label=f"S{s}", path_effects=stroke())
    phase_lines(ax[1], top=1.03)
    ax[1].set(xlabel="time from DELAY onset (s)", ylabel="P(state)", ylim=(-0.02, 1.06),
              title="occupancy (grey = WM delay)")
    ax[1].legend(fontsize=8, ncol=K, loc="lower right")
    flux = net_flux(res.trans_counts); vmax = np.abs(flux).max() or 1
    im = ax[2].imshow(flux, cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    for i in range(K):
        for jj in range(K):
            if i != jj: ax[2].text(jj, i, f"{flux[i,jj]:+d}", ha="center", va="center", fontsize=8)
    ax[2].set(xlabel="to state", ylabel="from state", xticks=range(K), yticks=range(K),
              title=f"net flux (fwd frac={res.forward_fraction:.2f})")
    fig.colorbar(im, ax=ax[2], label="net i->j")
    fig.tight_layout(); savefig(fig, PHASE, f"{SESSION}_{a}_hmm"); return fig
for a in AREAS:
    plot_hmm(a, results[a])
plt.show()

## Part 5 — Do the states track the task phase?

Label every bin with the task phase (from the per-trial event onsets), then measure how much the
decoded state agrees — referenced to a **trial-permutation null** (shuffle each trial's event times).
⚠️ Because this task's timing is **near-deterministic**, "tracks phase" ≈ "tracks time": the raw
association (Cramér's V) can be real while the null-referenced z is ~0 — i.e. the states tile the
epochs, but that tiling is explained by the clock, not by trial-specific timing. Both are reported.

In [ ]:
EPOCHS = PHASE_NAMES            # fixation, stimulus, delay, cue, response, reward
landmarks = events[:, 1:6]     # onsets of stimulus, delay, cue, response, reward (delay-relative)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, a in zip(axes, AREAS):
    res = results[a]; K = res.occupancy.shape[0]
    al = state_phase_alignment(res.states, bin_centers, landmarks, n_states=K, n_shuffles=200, seed=SEED)
    im = ax.imshow(al["p_state_given_phase"], aspect="auto", cmap=SEQUENTIAL, vmin=0, vmax=1)
    ax.set_xticks(range(len(EPOCHS)), EPOCHS, rotation=30, ha="right")
    ax.set_yticks(range(K), [f"S{i}" for i in range(K)]); ax.set_ylabel("HMM state")
    ax.set_title(f"{a}  P(state | phase)\nCramer's V={al['cramers_v']:.2f}  (null-z={al['nmi_z']:+.1f}, p={al['nmi_p']:.2f})")
    fig.colorbar(im, ax=ax, fraction=0.046)
    print(f"{a}: NMI={al['nmi']:.3f} null={al['nmi_null_mean']:.3f} z={al['nmi_z']:+.1f} "
          f"p={al['nmi_p']:.3f} Cramer's V={al['cramers_v']:.2f}")
fig.suptitle("States vs task phase (V = descriptive tiling; z = beyond the clock)", y=1.02)
fig.tight_layout(); savefig(fig, PHASE, f"{SESSION}_state_phase"); plt.show()

## Part 6 — Read-outs and what this shows

`forward_fraction` (directedness) and `switch_rate` (how often the population changes state).

In [ ]:
rows = []
for a in AREAS:
    res = results[a]
    rows.append({"area": a, "K": res.occupancy.shape[0], "n_units": tensors[a].shape[2],
                 "n_trials": n_trials, "forward_fraction": round(res.forward_fraction, 3),
                 "switch_rate": round(switch_rate(res.states), 2)})
df = pd.DataFrame(rows); (ROOT/"results"/"hmm").mkdir(parents=True, exist_ok=True)
df.to_csv(ROOT/"results"/"hmm"/f"monkey_{SESSION}_readouts.csv", index=False)
display(df)

**Reading (provisional — one session, one animal).**
- **DMFC** has a reproducible **K = 3** (interior ARI peak) and a **persistent state configuration held
  across fixation→stimulus→delay**, switching to a distinct state only at the **response/reward** — a
  working-memory-like *maintenance-then-transition* signature, now visible against the delay line.
- **FEF** is nearly **static within-trial** (switch rate ≈ 0): its states cluster *trials* by firing
  level rather than carving *time*. Do not read this as "FEF has no states" — its structure may be
  locked to gaze/saccade events not represented in these phase times.
- **State↔phase**: the states tile the epochs descriptively (Cramér's V), but because the task timing
  is near-deterministic, the trial-permutation null shows little *beyond-clock* effect.

**Next:** replicate across the 14 usable sessions (pool the statistics, per session × area), and add
the 2nd animal when it arrives; try a higher K if you want to separate stimulus-encoding from delay-
maintenance within the sustained regime.